# Conclusions — the proforma evidence pack

One notebook, one section per conclusion. Every section imports its analysis from
`conclusion/demo/` — the **same modules the Streamlit app imports** — so the notebook and the app
can never report different numbers.

| Section | Question | Analysis module | Input |
|---|---|---|---|
| ① Tunnel length | How do sites ramp, and do we use the tunnel we build? | `tunnel_data.py` | `conclusion/data/tunnel_length_with_wash.csv` |
| ② Proforma backtest | When the proforma projected a wash count, how close did it land? | `proforma_data.py` | `conclusion/data/n70_backtest_dataset.csv` |

> Runs in the `sonnys` conda env (`conda activate sonnys`), the same one the app runs in. Charts are
> Plotly because that env has plotly and does **not** have matplotlib.

---

# Section ① — Tunnel length, wash trajectories and year-5 volume

**One input file**, and only the sites with at least three years of trading — **39 of the 78** —
so every year-5 figure is either observed outright (33) or a short carry-forward (6). Each carries a
measured built tunnel length, measured peak hourly throughput and the full annual wash history.

**The work that matters is getting to a comparable year-5 number.** The file's wash columns are
*calendar* years, but sites open mid-year, so the first and last columns are part-years. Each is
scaled up by the fraction of the year the site was actually open and inside the data window, which
turns the series into **operating years** (year 1 = first 12 months trading). Sites younger than
five years are then carried to year 5 on a maturity ramp measured from the sites that do have five
years — and that ramp is validated by holdout in §1.2.

Two supporting facts, both derived from the data rather than assumed:

- **The extract ends mid-2026.** Across 34 mature sites trading through both years, 2026 washes are
  a median 0.539 of 2025 washes (IQR 0.52–0.56) — about 6.5 months of coverage.
- **Tunnel lengths are whole metres.** `tunnel_length_actual_ft` is metres × 3.2; dividing by 3.2
  returns exact integers 13–60. We convert at the true 3.28084 ft/m.

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "demo"))
import tunnel_data as td

INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"
S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"
STATUS = {"Overbuilt": "#d03b3b", "Right-sized": "#0ca30c", "At capacity": "#fab219"}
MAT_COLOR = {"5+ years observed": S1, "3–4 years observed": S3,
             "1–2 years observed": S2, "Stopped reporting": MUTED}

pio.templates["sonnys"] = go.layout.Template(layout=dict(
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color=INK2, size=12),
    title=dict(font=dict(color=INK, size=15)),
    xaxis=dict(gridcolor=GRID, zeroline=False, linecolor="#c3c2b7", ticks="outside",
               tickcolor=GRID, tickfont=dict(color=MUTED)),
    yaxis=dict(gridcolor=GRID, zeroline=False, linecolor="#c3c2b7", ticks="outside",
               tickcolor=GRID, tickfont=dict(color=MUTED)),
    legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(color=INK2)), margin=dict(l=70, r=30, t=60, b=60),
))
pio.templates.default = "sonnys"

d = td.build()
h = td.headline(d)

print(f"sites                    : {h['n_sites']}")
print(f"  with an observed year 5: {h['n_observed5']}")
print(d.maturity.value_counts().reindex(td.MATURITY_ORDER).to_string())
print(f"\nmedian year-5 washes     : {h['median_year5']:,.0f}")
print(f"median tunnel            : {h['median_tunnel_ft']:.0f} ft "
      f"({h['median_tunnel_ft']/td.FT_PER_M:.0f} m)")
print(f"median tunnel used       : {h['median_utilisation']:.1%}  (highest daily peak)")

sites                    : 39
  with an observed year 5: 33
maturity
5+ years observed     33
3–4 years observed     6

median year-5 washes     : 64,531
median tunnel            : 121 ft (37 m)
median tunnel used       : 63.7%  (highest daily peak)


## 1.1 How a car wash ramps up

Median annualised wash rate by operating year, on a **balanced panel** — only sites observed in
every one of their first five years. Without that restriction the curve moves as sites enter and
leave the sample, which reads as a decline that is really composition changing underneath it.

In [2]:
cc = td.cohort_curve()

fig = go.Figure()
fig.add_scatter(x=list(cc.operating_year) + list(cc.operating_year[::-1]),
                y=list(cc.p75) + list(cc.p25[::-1]), fill="toself",
                fillcolor="rgba(42,120,214,0.15)", line=dict(width=0), hoverinfo="skip",
                name="middle half of sites")
fig.add_scatter(x=cc.operating_year, y=cc["median"], mode="lines+markers", name="median site",
                line=dict(color=S1, width=3),
                marker=dict(size=10, line=dict(width=2, color=SURFACE)),
                hovertemplate="Operating year %{x}<br><b>%{y:,.0f} washes/yr</b><extra></extra>")
fig.update_layout(title=f"The ramp is over by year 2 ({int(cc.sites.iloc[0])} sites, balanced panel)",
                  xaxis_title="Operating year", yaxis_title="Annualised washes per year",
                  height=430, xaxis=dict(dtick=1),
                  legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0))
fig.show()

print(cc.round(3).to_string(index=False))

 operating_year    median       p25        p75  share_of_year5  sites
              1 61763.784 32026.257  76372.909           0.588     27
              2 94205.000 54966.500 115761.500           0.982     27
              3 91731.000 58212.500 139312.500           1.039     27
              4 84408.000 50622.500 131239.260           1.007     27
              5 85820.874 53838.497 127479.740           1.000     27


**Insights**

- **Reading.** The median site does **59% of its year-5 volume in year 1**, jumps to **98% by
  year 2**, and then goes flat — years 3, 4 and 5 sit within a few points of each other.
- **So-what.** The ramp is essentially a single step, not a five-year climb. A site that has traded
  **two full years is already showing its long-run number**, which is the earliest point a build
  decision can be judged on realised data rather than a projection.
- **Caveat.** Balanced panel of 27 sites, so this is the shape for sites that survived five years.
  The band is wide — the *shape* is consistent, the *level* varies several-fold between sites.

## 1.2 How reliable is the year-5 call?

Every site that actually reached year 5, predicted from each earlier year and checked against what
happened. `naive` assumes the site never grows again; `ramp` applies the §1.1 curve.

In [3]:
v = td.validation()

fig = go.Figure()
fig.add_bar(x=v.from_operating_year, y=v.mdape_naive, name="assume no further growth",
            marker=dict(color=MUTED, line=dict(width=2, color=SURFACE)),
            hovertemplate="from year %{x}<br>%{y:.1f}% error<extra></extra>")
fig.add_bar(x=v.from_operating_year, y=v.mdape_ramp, name="using the maturity ramp",
            marker=dict(color=S1, line=dict(width=2, color=SURFACE)),
            hovertemplate="from year %{x}<br>%{y:.1f}% error<extra></extra>")
fig.update_layout(title="Predicting year 5 from an earlier year — median absolute error",
                  xaxis_title="Predicting from operating year", yaxis_title="Median absolute % error",
                  height=400, barmode="group", xaxis=dict(dtick=1),
                  legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0))
fig.show()

print(v.round(2).to_string(index=False))
print("\nramp used:", td.RAMP)

 from_operating_year  sites  mdape_naive  mdape_ramp  bias
                   1     27        41.39       19.46  0.89
                   2     32        20.86       13.38  1.10
                   3     32         9.20        9.20  1.03
                   4     33         6.01        6.01  1.01

ramp used: {1: 0.67, 2: 0.88, 3: 1.0, 4: 1.0, 5: 1.0}


**Insights**

- **Reading.** From a **single year** of trading the ramp calls year 5 to within **19.5%**, against
  **41.4%** if you read year 1 at face value — the ramp removes more than half the error. By year 3
  the two converge at **9.2%**, because the site is already mature.
- **So-what.** A defensible year-5 number exists **from the first year of trading**. That is what
  lets the six sites here with only three or four years be carried to year 5 without guessing.
- **Caveat.** The ramp factors were fitted on these same sites, so 19.5% is an in-sample figure and
  the true error on a new site will be somewhat higher. It is also fitted on sites opened around
  2019 and applied to sites opened around 2023 — only 6 sites here rely on it. The bias column
  (0.89–1.03) shows no systematic over- or under-call.

## 1.3 Does a longer tunnel deliver more washes?

Year-5 volume against built length, coloured by how much real history is behind each point.

In [4]:
fit = td.length_vs_volume(d)

fig = go.Figure()
for m in td.MATURITY_ORDER:
    g = d[(d.maturity == m) & d.year5_washes.notna()]
    if g.empty:
        continue
    fig.add_scatter(x=g.tunnel_ft, y=g.year5_washes, mode="markers", name=m,
                    marker=dict(size=11, color=MAT_COLOR[m], line=dict(width=1.6, color=SURFACE)),
                    customdata=g[["site", "utilisation"]],
                    hovertemplate="<b>%{customdata[0]}</b><br>%{x:.0f} ft · %{y:,.0f} washes/yr"
                                  "<br>%{customdata[1]:.0%} of capacity<extra></extra>")
xs = np.linspace(d.tunnel_ft.min(), d.tunnel_ft.max(), 50)
fig.add_scatter(x=xs, y=fit["slope"]*xs + fit["intercept"], mode="lines",
                name=f"+{fit['slope']:,.0f} washes per foot",
                line=dict(color=INK2, width=2, dash="dash"), hoverinfo="skip")
fig.update_layout(title="Longer tunnels wash more — but length explains a fifth of the spread",
                  xaxis_title="Built tunnel length (ft)", yaxis_title="Year-5 washes per year",
                  height=470, legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0,
                                          font=dict(size=10)))
fig.add_annotation(x=1, y=0, xref="paper", yref="paper", xanchor="right", showarrow=False,
                   text=f"n = {fit['n']}   r = {fit['r']:.2f}   R² = {fit['r2']:.2f}",
                   font=dict(color=MUTED, size=11))
fig.show()

print(d.groupby("tier", observed=True).agg(
    sites=("site_key", "size"), median_ft=("tunnel_ft", "median"),
    median_year5=("year5_washes", "median"),
    median_best_hour=("peak_cars_per_hour", "median"),
    capacity_used=("utilisation", "median")).round(2).to_string())

            sites  median_ft  median_year5  median_best_hour  capacity_used
tier                                                                       
<100 ft         7      91.86      59889.00              59.0           0.64
100–120 ft     12     113.19      48450.80              63.5           0.58
120–140 ft     11     124.67     134820.63              83.0           0.67
140 ft+         9     147.64     117119.00              93.0           0.63


**Insights**

- **Reading.** **+1,337 washes per foot** (r = 0.49), but **R² = 0.24** — three-quarters of the
  difference between sites is something other than how long the tunnel is.
- **Reading (bands).** Across the four length bands, **capacity used barely moves** — 64%, 58%,
  67%, 63%. Volume and capacity rise together, so extra feet buy headroom rather than throughput.
- **So-what.** Length is a weak lever on volume. The shortest band (<100 ft) runs at the same 64%
  as the longest, so nothing is gained in utilisation by building bigger.
- **Caveat.** n = 39, with 9–12 sites per band, so read the band medians as directional.

## 1.4 How much of the tunnel do we use?

The proforma sizes a tunnel at **one foot per car per hour** of year-5 peak volume. This is each
site's *measured* highest daily peak against that rating — the most generous test available to the tunnel.

In [5]:
med = d.utilisation.median()

fig = go.Figure()
fig.add_histogram(x=d.utilisation, nbinsx=20,
                  marker=dict(color=S1, line=dict(width=1.2, color=SURFACE)),
                  hovertemplate="%{x:.0%} of capacity<br>%{y} sites<extra></extra>")
fig.add_vline(x=med, line=dict(color=INK, width=2), annotation_text=f"median {med:.0%}",
              annotation_position="top left", annotation_font=dict(color=INK, size=12))
fig.add_vline(x=1.0, line=dict(color=STATUS["At capacity"], width=2, dash="dash"),
              annotation_text="full capacity", annotation_position="top right",
              annotation_font=dict(color=STATUS["At capacity"], size=11))
fig.update_layout(title="No site has ever run out of tunnel",
                  xaxis_title="Share of the tunnel used at its highest daily peak",
                  yaxis_title="Sites", height=430, showlegend=False, xaxis=dict(tickformat=".0%"))
fig.show()

bb = td.utilisation_by_basis()
print(bb.round(3).to_string(index=False))
print(f"\nsites that have ever exceeded their rating: {(d.utilisation > 1).sum()}")

             basis   p25  median   p75   max  pct_overbuilt
 Median daily peak 0.148   0.242 0.341 0.521          0.949
    p75 daily peak 0.197   0.315 0.437 0.640          0.795
    p90 daily peak 0.247   0.398 0.528 0.747          0.718
Highest daily peak 0.486   0.637 0.734 0.975          0.282

sites that have ever exceeded their rating: 0


**Insights**

- **Reading.** On the highest daily peak each site has ever recorded, the median uses **64%** of its
  rated capacity and the busiest site reaches **98%** — **no site has ever exceeded its rating.** On
  a median day the figure is **24%**.
- **So-what.** Throughput is not the binding constraint anywhere in this estate; demand is. The
  tunnel is sized for an hour that never arrives.
- **Caveat.** The daily peak is a throughput measure, not a queueing one — arrivals bunch up,
  so some headroom is legitimate. That argues for a margin, not for the 58% of tunnel length that
  §1.5 shows is spare.

## 1.5 Where the spare tunnel is

For the sites furthest from their rating: the length their own highest daily peak calls for,
against the length that was built. Floored at the shortest tunnel anyone in the sample built (13 m).

In [6]:
over = d[d.verdict == "Overbuilt"].nlargest(15, "excess_ft")
labels = [f"{i}. {str(r.site)[:26]}" for i, r in enumerate(over.itertuples(), 1)]

fig = go.Figure()
fig.add_bar(y=labels, x=over.required_ft, orientation="h", name="length its peak hour needs",
            marker=dict(color=S1, line=dict(width=1.5, color=SURFACE)),
            hovertemplate="%{y}<br>needs %{x:.0f} ft<extra></extra>")
fig.add_bar(y=labels, x=over.excess_ft, orientation="h", name="spare length",
            marker=dict(color=STATUS["Overbuilt"], line=dict(width=1.5, color=SURFACE)),
            hovertemplate="%{y}<br>%{x:.0f} ft spare<extra></extra>")
fig.update_layout(title="The 15 largest gaps between the tunnel needed and the tunnel built",
                  barmode="stack", xaxis_title="Tunnel length (ft)", height=560,
                  yaxis=dict(autorange="reversed", showgrid=False),
                  legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0))
fig.show()

print(f"sites under half their rating : {h['n_overbuilt']} of {h['n_sites']}")
print(f"median spare length           : {h['median_excess_ft']:.0f} ft "
      f"({h['median_excess_ft']/td.FT_PER_M:.0f} m)")
print(f"spare share of the tunnel     : {h['median_excess_share']:.0%}")

sites under half their rating : 11 of 39
median spare length           : 72 ft (22 m)
spare share of the tunnel     : 58%


**Insights**

- **Reading.** **11 of 39** sites never reach half their rated capacity even on their best day. The
  median one carries **72 ft (22 m) of spare tunnel — 58% of its length**.
- **So-what.** These are the builds to re-read before the next site is specced on the same template;
  the same traffic would have gone through a materially shorter tunnel.
- **Caveat.** Stated in feet, not dollars, on purpose — this dataset carries no cost data, so any
  dollar figure would have to come from the build team's cost per foot.

## Section ① conclusions

1. **The ramp is a single step, not a climb.** The median site does **59%** of its year-5 volume in
   year 1 and **98%** by year 2, then goes flat.

2. **Year 5 can be called from year 1 to within ~19%** using that ramp, against 41% if the first
   year is read at face value.

3. **Length is a weak driver of volume.** +1,337 washes per foot, but **R² = 0.24**.

4. **No site has ever run out of tunnel.** Median utilisation **64%** on the highest daily peak ever
   recorded, maximum **98%**, and **24%** on a median day.

5. **11 of 39 sites carry a median 58% of their tunnel as spare** — 72 ft that their own best day
   never calls for.

---

# Section ② — Proforma backtest: when we projected a wash count, how close did it land?

**Question.** Every one of these sites was underwritten on an Excel proforma that projected a wash
count. They were then built, and traded long enough to score. How close was the projection — and
does our model do better on the same sites?

**Data.** `proforma/data/conclusion/n70_backtest_dataset.csv` — one row per site for the 70 mature,
matched sites, carrying every proforma input, all three forecasters, and the actual washes side by
side. Actuals are recomputed live from the panel and reconciled cell-for-cell (0 mismatches); see
that folder's README.

**Reference.** `experiments/old-proforma-analysis/proforma_backtest.ipynb` is the working version of
this analysis; this section is the reviewed cut of it, sharing code with the app.

In [7]:
import proforma_data as pf

FCOLOR = {"proforma": S2, "coldstart": S3, "model5": S1}
GOOD, WARN, BAD = "#0ca30c", "#fab219", "#d03b3b"

pdf = pf.load()                      # collapse cases dropped — see the caveat in 2.5
ph = pf.headline(pdf)
sc = pf.scorecard(pdf)

print(f"sites backtested : {ph['n_sites']}  (of 70; 2 collapsed sites excluded)")
print(f"proforma         : {ph['proforma_mdape']:.1f}% median error, bias {ph['proforma_bias']:.2f}x")
print(f"model 5 (LOSO)   : {ph['model5_mdape']:.1f}% median error, bias {ph['model5_bias']:.2f}x")
print(f"model 5 closer on: {ph['win_rate']*100:.0f}% of sites (p={ph['win_p']:.1e})")

sites backtested : 68  (of 70; 2 collapsed sites excluded)
proforma         : 58.5% median error, bias 1.37x
model 5 (LOSO)   : 29.4% median error, bias 0.98x
model 5 closer on: 72% of sites (p=3.6e-04)


## 2.1 Every projection against what actually happened

No modelling. One dot per site: the washes the proforma projected against the washes that showed up.
Anything above the dashed line was over-projected.

In [8]:
a, pr = pdf.actual_mature_wash, pdf.proforma_y5
over = pr > a
lim = [0, float(max(a.max(), pr.max())) * 1.05]

fig = go.Figure()
fig.add_scatter(x=lim, y=lim, mode="lines", name="perfect projection",
                line=dict(color=INK2, width=2, dash="dash"), hoverinfo="skip")
for mask, name, col in [(over, "Over-projected", BAD), (~over, "Under-projected", S1)]:
    g = pdf[mask]
    fig.add_scatter(x=g.actual_mature_wash, y=g.proforma_y5, mode="markers", name=name,
                    marker=dict(size=11, color=col, line=dict(width=1.6, color=SURFACE)),
                    customdata=g[["client_name", "state"]].fillna("-"),
                    hovertemplate="<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
                                  "actual %{x:,.0f}/mo<br>proforma %{y:,.0f}/mo<extra></extra>")
fig.update_layout(title="The proforma sits above the line on nearly three sites in four",
                  xaxis_title="Actual mature washes per month",
                  yaxis_title="Proforma projection per month", height=520,
                  xaxis=dict(range=lim, constrain="domain"),
                  yaxis=dict(range=lim, scaleanchor="x", constrain="domain"),
                  legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0))
fig.show()

print(f"over-projected : {ph['proforma_over_share']*100:.0f}% of sites")
print(f"median ratio   : {ph['proforma_bias']:.2f}x actual")
print(f"p90 ratio      : {ph['proforma_p90']:.2f}x actual")

over-projected : 72% of sites
median ratio   : 1.37x actual
p90 ratio      : 4.33x actual


**Insights**

- **Reading.** **72% of sites sit above the line.** The median proforma projects **1.37× the washes
  that showed up**, and at the 90th percentile it projects **4.33×** reality.
- **So-what.** This is a systematic optimism bias, not scatter. A capital plan built on these
  numbers over-commits on roughly three sites in four, and the worst cases are over by multiples
  rather than percentages.
- **Caveat.** n = 68, and these are sites that were *both* proforma'd and built — proposals killed
  at the proforma stage are invisible. If optimistic proformas were more likely to get approved,
  this overstates the bias of the proforma *process* while still correctly describing what got built.

## 2.2 Does the projection get better as the site matures?

The proforma exists to underwrite years 4-5, when the debt is being serviced. So the error in those
years matters more than the error in year 1.

In [9]:
by = pf.by_year(pdf, full_years_only=True)

fig = go.Figure()
for key, label, _, _, _ in pf.FORECASTERS:
    g = by[by.key == key]
    fig.add_scatter(x=g.year, y=g.mdape, mode="lines+markers", name=label,
                    line=dict(color=FCOLOR[key], width=2),
                    marker=dict(size=9, line=dict(width=1.5, color=SURFACE)),
                    customdata=g[["n"]],
                    hovertemplate=f"<b>{label}</b><br>year %{{x}}<br>%{{y:.1f}}% error<br>"
                                  "n = %{customdata[0]}<extra></extra>")
fig.update_layout(title="The proforma's error is flat-to-worsening; the model's tightens",
                  xaxis_title="Operating year", yaxis_title="Median absolute % error",
                  height=440, xaxis=dict(dtick=1),
                  legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0))
fig.show()

print(by.pivot(index="year", columns="forecaster", values="mdape").round(1).to_string())
print()
print(by.pivot(index="year", columns="forecaster", values="bias").round(2).to_string())

forecaster  Cold-start v15  Model 5 (ensemble)  Proforma (Excel)
year                                                            
1                     48.7                42.4              45.9
2                     41.6                39.7              46.6
3                     35.6                34.7              46.9
4                     38.6                23.4              47.1
5                     47.3                30.5              54.8

forecaster  Cold-start v15  Model 5 (ensemble)  Proforma (Excel)
year                                                            
1                     1.41                1.00              1.12
2                     1.27                0.99              1.13
3                     1.25                0.94              1.18
4                     1.20                1.04              1.24
5                     1.39                1.05              1.42


**Insights**

- **Reading.** The proforma runs **45.9% error in year 1 and 54.8% by year 5** — flat, then worse.
  Model 5 moves the other way, tightening from 42.4% to **30.5%**. The proforma's bias also grows
  with age (1.12× → 1.42×).
- **So-what.** The proforma is not "roughly right, eventually". Its miss is structural and it is
  **worst in exactly the years it exists to underwrite** — the ones carrying the debt service.
- **Caveat.** Year 4-5 rest only on sites old enough to have them (n falls from 63 to 20), and those
  are the earliest cohorts — a survivorship cut, not a random sample. Partial years are excluded.

## 2.3 Scorecard — three forecasters, same sites, same target

Model 5 is scored **leave-one-site-out**: each site is predicted by a model fitted without it, so
this is out-of-sample, not a fit.

In [10]:
fig = go.Figure(go.Bar(
    x=sc.forecaster, y=sc.mdape,
    marker=dict(color=[FCOLOR[k] for k in sc.key], line=dict(width=2, color=SURFACE)),
    text=[f"{v:.1f}%" for v in sc.mdape], textposition="outside",
    textfont=dict(color=INK, size=13),
    hovertemplate="%{x}<br>%{y:.1f}% median error<extra></extra>"))
fig.update_layout(title="Median absolute error against actual mature washes",
                  yaxis_title="Median absolute % error", height=420,
                  yaxis=dict(range=[0, sc.mdape.max()*1.25]), xaxis=dict(showgrid=False))
fig.show()

print(sc[["forecaster","n","mdape","bias","over_share","within_25","p10","p90","iqr"]]
      .round(3).to_string(index=False))
h2h = pf.head_to_head(pdf)
print(f"\nmodel5 closer than proforma on {h2h['wins']}/{h2h['n']} sites "
      f"({h2h['win_rate']*100:.1f}%)  sign-test p={h2h['binom_p']:.2e}  "
      f"wilcoxon p={h2h['wilcoxon_p']:.2e}")

        forecaster  n  mdape  bias  over_share  within_25   p10   p90   iqr
  Proforma (Excel) 68 58.451 1.374       0.721      0.294 0.644 4.328 1.491
    Cold-start v15 68 46.725 1.366       0.735      0.279 0.675 2.659 0.968
Model 5 (ensemble) 68 29.438 0.985       0.500      0.412 0.410 1.943 0.584

model5 closer than proforma on 49/68 sites (72.1%)  sign-test p=3.58e-04  wilcoxon p=2.05e-05


**Insights**

- **Reading.** Proforma **58.5%** vs Model 5 **29.4%** median error — the model roughly **halves the
  miss** — and Model 5 is near-unbiased (**0.98×**) where the proforma runs **1.37×**.
- **So-what.** Model 5 is closer on **49 of 68 sites (72%)**, and that is not luck: sign test
  p = 3.6e-04, Wilcoxon p = 2.1e-05. Since it is leave-one-site-out, this is what you would expect
  on the next site, not an in-sample flatter.
- **Caveat.** Even the best forecaster misses ~29% at the median and lands within ±25% on only 41%
  of sites. "Materially better", not "solved" — and cold-start (46.7%) shows roughly half the gain
  comes from location alone, before any build detail.

## 2.4 The proforma scores a site on 10 factors — which actually predict?

Spearman rank correlation of each factor's own score against the washes that showed up. Of the 15 candidate inputs, 14 are testable — *weekly hours* is the same choice at every
site, so it cannot correlate with anything. With 14 tests on 68 sites one "significant" result
would appear by chance, so a Benjamini-Hochberg
false-discovery correction is applied across the whole family.

In [11]:
ft = pf.factor_table(pdf)
cols = [GOOD if s == "yes" else WARN if s == "marginal" else MUTED for s in ft.signif]

fig = go.Figure(go.Bar(
    x=ft.rho, y=ft.factor, orientation="h",
    marker=dict(color=cols, line=dict(width=1.6, color=SURFACE)),
    customdata=ft[["kind", "p", "q"]],
    hovertemplate="<b>%{y}</b> (%{customdata[0]})<br>rho %{x:.3f}<br>"
                  "p %{customdata[1]:.3f} · q %{customdata[2]:.3f}<extra></extra>"))
fig.add_vline(x=0, line=dict(color=INK2, width=1.5))
fig.update_layout(title="Only the capacity factors survive — green passes FDR, grey does not",
                  xaxis_title="Spearman correlation with actual mature washes", height=520,
                  yaxis=dict(autorange="reversed", showgrid=False))
fig.show()

print(ft.round(3).to_string(index=False))

                factor        kind  levels    rho     p     q signif
          pay stations Site factor       4  0.432 0.000 0.003    yes
          type of site Site factor       5  0.369 0.002 0.014    yes
     free vacuum slots Site factor       4  0.368 0.002 0.009    yes
 cumulative site score     Roll-up      23  0.344 0.004 0.014    yes
         traffic count      Market      68  0.218 0.075 0.209     no
    avg household size      Market      24  0.177 0.150 0.349     no
     entrance stack up Site factor       4  0.129 0.293 0.586     no
          area profile Site factor       3  0.092 0.454 0.707     no
            visibility Site factor       4  0.072 0.558 0.781     no
         traffic speed Site factor       4  0.062 0.614 0.716     no
    site accessibility Site factor       3  0.044 0.719 0.719     no
   nearest competition Site factor       4 -0.061 0.619 0.667     no
     % hh income $35k+      Market      63 -0.065 0.596 0.758     no
cumulative demographic     Roll-up

**Insights**

- **Reading.** Only **4 of the 14 testable** inputs survive the FDR correction — **pay stations (+0.43), type of
  site (+0.37), free vacuum slots (+0.37)** and the cumulative site score they drive (+0.35). Every
  survivor is a **capacity** variable: how many cars the site can physically process.
- **So-what.** Competition, visibility, accessibility, traffic speed and the whole demographic block
  sit at |rho| < 0.15 — and the cumulative demographic score is actually **negative (−0.11)**. The
  proforma weights all ten factors equally-ish; the evidence supports weighting about three.
- **Caveat.** Univariate rank correlations on n = 68: this ranks which inputs carry signal, not how
  much each adds independently, and the surviving capacity factors are correlated with each other.
  Confirms the prior finding recorded for the 121-proforma backtest — this is an independent cut of
  the same conclusion, not new evidence for it.

## 2.5 Two different ways to be wrong

Bias and spread fail differently, and only one of them can be fixed with a haircut.

In [12]:
fig = go.Figure()
for key, label, _, mature_col, _ in pf.FORECASTERS:
    ratio = (pdf[mature_col] / pdf.actual_mature_wash).replace([np.inf, -np.inf], np.nan).dropna()
    fig.add_box(x=ratio.clip(upper=6), name=label, marker_color=FCOLOR[key],
                boxpoints="all", jitter=0.5, pointpos=0, line=dict(width=2),
                marker=dict(size=6, opacity=0.6),
                hovertemplate=f"<b>{label}</b><br>%{{x:.2f}}x actual<extra></extra>")
fig.add_vline(x=1.0, line=dict(color=INK2, width=2, dash="dash"),
              annotation_text="perfect", annotation_position="top right",
              annotation_font=dict(color=INK2, size=11))
fig.update_layout(title="Predicted / actual per site — the proforma is off-centre AND wide",
                  xaxis_title="Predicted / actual (clipped at 6x for display)", height=460,
                  yaxis=dict(showgrid=False), showlegend=False)
fig.show()

pr_ = sc.set_index("key")
for k in ["proforma", "coldstart", "model5"]:
    print(f"{pr_.loc[k,'forecaster']:20s} median {pr_.loc[k,'bias']:.2f}x  "
          f"p10 {pr_.loc[k,'p10']:.2f}x  p90 {pr_.loc[k,'p90']:.2f}x  IQR {pr_.loc[k,'iqr']:.2f}")

Proforma (Excel)     median 1.37x  p10 0.64x  p90 4.33x  IQR 1.49
Cold-start v15       median 1.37x  p10 0.67x  p90 2.66x  IQR 0.97
Model 5 (ensemble)   median 0.98x  p10 0.41x  p90 1.94x  IQR 0.58


**Insights**

- **Reading.** The proforma is both off-centre and wide — median **1.37×** with a p10-p90 span of
  **0.64×-4.33×**. Model 5 is centred (**0.98×**) and about **2.5× tighter** (IQR 0.58 vs 1.49).
- **So-what.** A flat haircut cannot fix this. Shaving 27% off every proforma would centre the
  median and still leave the p90 site over-projected by more than 3×.
- **Caveat.** Ratios are clipped at 6× for display only; every statistic uses unclipped values. Two
  collapsed sites are excluded throughout — including them would make the proforma look *worse*
  (one reads as a 205× miss), but they are closures, not forecast misses.

## 2.6 The tunnel length that formula produces

The proforma sizes the tunnel straight off its own year-5 peak projection — **one foot of tunnel per
car per hour, with no +20 added**. Verified against this file: `tunnel_length_ft` is *exactly*
`year5_max_hourly`. `tunnel_length_actual_m` is the tunnel that was really built, for 61 of the 70.

In [13]:
tl = pf.tunnel_lengths(pdf)
ts = pf.tunnel_length_stats(pdf)
lim = [0, float(max(tl.actual_m.max(), tl.formula_m.max())) * 1.05]

fig = go.Figure()
fig.add_scatter(x=lim, y=lim, mode="lines", name="formula matches the build",
                line=dict(color=INK2, width=2, dash="dash"), hoverinfo="skip")
for mask, name, col in [(tl.gap_m > 0, "Built longer than the formula", BAD),
                        (tl.gap_m <= 0, "Built shorter than the formula", S1)]:
    g = tl[mask]
    fig.add_scatter(x=g.actual_m, y=g.formula_m, mode="markers", name=name,
                    marker=dict(size=11, color=col, line=dict(width=1.6, color=SURFACE)),
                    customdata=g[["client_name", "gap_m"]],
                    hovertemplate="<b>%{customdata[0]}</b><br>built %{x:.0f} m · "
                                  "formula %{y:.0f} m<br>%{customdata[1]:+.0f} m<extra></extra>")
fig.update_layout(title="The formula asks for a shorter tunnel than the one that gets built",
                  xaxis_title="Tunnel actually built (m)",
                  yaxis_title="Length the formula calls for (m)", height=480,
                  xaxis=dict(range=lim, constrain="domain"),
                  yaxis=dict(range=lim, scaleanchor="x", constrain="domain"),
                  legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0))
fig.show()

for k, v in ts.items():
    print(f"  {k:20s} {round(v, 3) if isinstance(v, float) else v}")
print()
print(pf.length_signal_check(pdf).round(3).to_string(index=False))

  n                    59
  r                    0.238
  p                    0.069
  rho                  0.222
  p_rho                0.091
  median_actual        38.0
  median_formula       29.017
  median_gap           6.314
  mae                  12.177
  median_ratio         0.792
  built_longer_share   0.678

            measure                     tracks   rho     p
     Formula length       Actual mature washes 0.455 0.000
     Formula length Proforma year-5 projection 0.997 0.000
     Formula length        Actual built length 0.222 0.091
Actual built length       Actual mature washes 0.302 0.020
Actual built length Proforma year-5 projection 0.216 0.101


**Insights**

- **Reading.** The formula length correlates **0.997** with the proforma's own year-5 volume
  projection but only **0.22** with the tunnel that actually got built — a typical miss of **12 m**
  on tunnels of 20–61 m. It is the volume number in different units, not a length estimate.
- **Reading (direction).** It also asks for *less* tunnel than gets built: median **29 m against
  38 m actually built**, with **68%** of sites built longer than the formula calls for.
- **So-what.** Two errors compound. The formula inherits the §2.1 volume bias — a projection running
  1.37× hot produces a length running 1.37× hot — and then the build ignores it anyway and goes
  longer still. Section ① then shows the result: no site has ever used more than 98% of the tunnel
  it ended up with.
- **Caveat.** n = 59 sites carry both a formula length and a measured build. The 0.997 correlation
  is close to mechanical — the formula is a linear transform of `year5_max_hourly` — which is
  exactly the point being made, not a finding about the world.

## Section ② conclusions

1. **The proforma over-projects, systematically.** Median **1.37×** actual, over on **72%** of
   sites, **4.33×** at the 90th percentile. Median absolute error **58.5%**.

2. **It does not improve with maturity — it degrades.** 45.9% error in year 1, **54.8% by year 5**,
   with bias growing 1.12× → 1.42×. The years it exists to underwrite are the ones it gets most wrong.

3. **A model roughly halves the miss.** Model 5 scores **29.4%** median error at **0.98×** bias on
   the identical sites, closer on **72%** of them (p = 3.6e-04) — and it is leave-one-site-out, so
   that is an out-of-sample result.

4. **Only capacity inputs carry signal.** Of the 14 testable proforma inputs, **4 survive FDR** — pay stations,
   type of site, free vacuum slots, and the site score they drive. Demographics and competition are
   flat-to-negative.

5. **You cannot fix it with a haircut.** The proforma is off-centre *and* 2.5× wider than the model,
   so any flat correction that fixes the median still leaves the tail badly over-projected.

6. **The tunnel-length formula is the volume projection restated.** It tracks the proforma's own
   year-5 number at **0.997** and the tunnel actually built at **0.22**. It asks for a median 29 m
   where 38 m gets built, and **68%** of sites are built longer than it calls for.

### What would sharpen it

- **n = 68 is the binding constraint**, and the set is conditioned on being built. Recovering
  proformas for sites that were *rejected* would separate the proforma's own bias from selection.
- **Re-weight rather than re-build.** The cheapest actionable change is dropping the ten-factor
  equal weighting for the three factors that survive — testable directly on this dataset.